# Notebook 09 — Final Error Analysis

**This notebook uses the final selected model and the reserved test set.**

Objectives:
1. Examine residuals on the test set
2. Segment customers by actual future value and compute per-segment metrics
3. Investigate systematic prediction patterns (underprediction of high-value customers)
4. Document model limitations honestly

In [ ]:
import sys
from pathlib import Path
PROJECT_ROOT = Path('..').resolve()
sys.path.insert(0, str(PROJECT_ROOT))
sys.path.insert(0, str(PROJECT_ROOT / 'src'))
import src as _src_pkg
if 'ecommerce_customer_value_prediction' not in sys.modules:
    sys.modules['ecommerce_customer_value_prediction'] = _src_pkg
import warnings; warnings.filterwarnings('ignore')
import json, math, numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
from scipy import stats
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:,.4f}'.format)
from ecommerce_customer_value_prediction.config import PROCESSED_DATA_DIR, MODELS_DIR, METRICS_DIR, FIGURES_DIR
MODELS_DIR.mkdir(parents=True, exist_ok=True)
METRICS_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
from ecommerce_customer_value_prediction.models.train import (
    NUMERIC_FEATURES, CATEGORICAL_FEATURES, TARGET_COL,
    build_pipeline, create_model, train, save_model, load_model, MODEL_PATHS,
    FINAL_MODEL_PATH, FINAL_METADATA_PATH, SUPPORTED_MODELS,
)
from ecommerce_customer_value_prediction.models.evaluate import evaluate
from ecommerce_customer_value_prediction.models.predict import predict
from sklearn.model_selection import train_test_split, KFold, cross_validate, cross_val_predict, GridSearchCV
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, make_scorer
CUSTOMER_CSV = PROCESSED_DATA_DIR / 'customer_dataset.csv'
df = pd.read_csv(CUSTOMER_CSV)
df['Customer ID'] = df['Customer ID'].astype('Int64')
RANDOM_STATE = 42; TEST_SIZE = 0.20; CV_FOLDS = 5
X = df[NUMERIC_FEATURES + CATEGORICAL_FEATURES].copy()
y = df[TARGET_COL].copy()
y_raw = df['future_revenue'].copy()
y_bins = pd.cut(y_raw, bins=[-0.01, 0.01, 50, 500, float('inf')], labels=['zero','low','medium','high'])
X_train, X_test, y_train, y_test, y_raw_train, y_raw_test = train_test_split(
    X, y, y_raw, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y_bins)
kf = KFold(n_splits=CV_FOLDS, shuffle=True, random_state=RANDOM_STATE)
def rmse(y_true, y_pred): return np.sqrt(mean_squared_error(y_true, y_pred))
scoring = {'mae': make_scorer(mean_absolute_error), 'rmse': make_scorer(rmse), 'r2': make_scorer(r2_score)}
print(f'Dataset: {df.shape} | Train: {len(X_train):,} | Test: {len(X_test):,}')
print(f'Modules loaded. RANDOM_STATE={RANDOM_STATE}')

In [ ]:
# Load final model
final_pipe = load_model(FINAL_MODEL_PATH)
print(f'Final model loaded: {FINAL_MODEL_PATH}')
print(f'Regressor: {type(final_pipe.named_steps["regressor"]).__name__}')
print()

# Generate test-set predictions
y_test_pred_log = final_pipe.predict(X_test)
y_test_actual_log = y_test.values
residuals_log = y_test_actual_log - y_test_pred_log

# Revenue scale
y_test_actual_gbp = np.expm1(y_test_actual_log)
y_test_pred_gbp   = np.expm1(y_test_pred_log)
residuals_gbp     = y_test_actual_gbp - y_test_pred_gbp
abs_err_gbp       = np.abs(residuals_gbp)

print(f'Test set: {len(X_test):,} customers')
print(f'MAE  (log): {mean_absolute_error(y_test_actual_log, y_test_pred_log):.4f}')
print(f'RMSE (log): {np.sqrt(mean_squared_error(y_test_actual_log, y_test_pred_log)):.4f}')
print(f'R²   (log): {r2_score(y_test_actual_log, y_test_pred_log):.4f}')
print(f'MAE  (£)  : £{mean_absolute_error(y_test_actual_gbp, y_test_pred_gbp):,.2f}')
print(f'RMSE (£)  : £{np.sqrt(mean_squared_error(y_test_actual_gbp, y_test_pred_gbp)):,.2f}')
print(f'Median AE (£): £{np.median(abs_err_gbp):,.2f}')

---
## Section 1 — Residual Diagnostic Plots

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 11))

# 1. Actual vs Predicted (log scale)
axes[0,0].scatter(y_test_pred_log, y_test_actual_log, alpha=0.25, s=15, color='steelblue')
mn, mx = min(y_test_pred_log.min(), y_test_actual_log.min()), max(y_test_pred_log.max(), y_test_actual_log.max())
axes[0,0].plot([mn,mx],[mn,mx],'r--',lw=1.5,label='Perfect')
axes[0,0].set_xlabel('Predicted log1p(revenue)'); axes[0,0].set_ylabel('Actual log1p(revenue)')
axes[0,0].set_title('Actual vs Predicted (log scale)'); axes[0,0].legend()

# 2. Residual vs Predicted (log scale)
axes[0,1].scatter(y_test_pred_log, residuals_log, alpha=0.25, s=15, color='darkorange')
axes[0,1].axhline(0, color='black', lw=1.5, linestyle='--')
axes[0,1].set_xlabel('Predicted log1p(revenue)'); axes[0,1].set_ylabel('Residual (actual - predicted)')
axes[0,1].set_title('Residual vs Predicted (log scale)')

# 3. Residual distribution
axes[1,0].hist(residuals_log, bins=60, color='mediumseagreen', edgecolor='white', density=True)
xr = np.linspace(residuals_log.min(), residuals_log.max(), 200)
axes[1,0].plot(xr, stats.norm.pdf(xr, residuals_log.mean(), residuals_log.std()), 'r-', lw=2, label='Normal fit')
axes[1,0].axvline(0, color='black', lw=1.5, linestyle='--')
axes[1,0].set_xlabel('Residual'); axes[1,0].set_ylabel('Density')
axes[1,0].set_title('Residual Distribution (log scale)'); axes[1,0].legend()

# 4. Absolute error distribution (£)
axes[1,1].hist(np.clip(abs_err_gbp, 0, np.percentile(abs_err_gbp, 95)), bins=60,
               color='crimson', edgecolor='white')
axes[1,1].axvline(np.median(abs_err_gbp), color='black', lw=2, linestyle='--',
                  label=f'Median=£{np.median(abs_err_gbp):.0f}')
axes[1,1].set_xlabel('Absolute Error £'); axes[1,1].set_ylabel('Count')
axes[1,1].set_title('Absolute Error Distribution £ (clipped at 95th percentile)'); axes[1,1].legend()

plt.suptitle('Final Model — Test Set Residual Diagnostics', fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig(PROJECT_ROOT / 'reports/figures/final_residual_diagnostics.png', dpi=150, bbox_inches='tight')
plt.show(); print('Saved: reports/figures/final_residual_diagnostics.png')

---
## Section 2 — Customer Segmentation Analysis

We segment test customers by **actual** future revenue and compute per-segment error metrics.
This reveals whether the model systematically underperforms on any customer group.

In [ ]:
# Create segments based on actual future revenue
test_analysis = pd.DataFrame({
    'actual_log': y_test_actual_log,
    'predicted_log': y_test_pred_log,
    'residual_log': residuals_log,
    'actual_gbp': y_test_actual_gbp,
    'predicted_gbp': y_test_pred_gbp,
    'residual_gbp': residuals_gbp,
    'abs_error_gbp': abs_err_gbp,
})

test_analysis['segment'] = pd.cut(
    test_analysis['actual_gbp'],
    bins=[-0.01, 0.01, 100, 500, 2000, float('inf')],
    labels=['Zero (churned)', 'Low (<£100)', 'Medium (£100-500)', 'High (£500-2000)', 'Very High (>£2000)']
)

seg_stats = test_analysis.groupby('segment', observed=True).agg(
    count=('actual_gbp', 'count'),
    mean_actual_gbp=('actual_gbp', 'mean'),
    mean_predicted_gbp=('predicted_gbp', 'mean'),
    mae_gbp=('abs_error_gbp', 'mean'),
    rmse_gbp=('abs_error_gbp', lambda x: np.sqrt((x**2).mean())),
    mean_residual_gbp=('residual_gbp', 'mean'),
    median_residual_gbp=('residual_gbp', 'median'),
).reset_index()

print('Per-segment error analysis (test set):')
print(seg_stats.to_string(index=False))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 5))
segs = seg_stats['segment'].astype(str).tolist()
x = np.arange(len(segs))

axes[0].bar(x, seg_stats['mae_gbp'], color='steelblue')
axes[0].set_xticks(x); axes[0].set_xticklabels(segs, rotation=25, ha='right')
axes[0].set_ylabel('MAE £'); axes[0].set_title('MAE by Customer Segment')

axes[1].bar(x, seg_stats['mean_residual_gbp'], color=['green' if v > 0 else 'red' for v in seg_stats['mean_residual_gbp']])
axes[1].axhline(0, color='black', lw=1.0)
axes[1].set_xticks(x); axes[1].set_xticklabels(segs, rotation=25, ha='right')
axes[1].set_ylabel('Mean Residual £'); axes[1].set_title('Mean Residual by Segment\n(positive = underprediction)')

axes[2].bar(x, seg_stats['count'], color='gray')
axes[2].set_xticks(x); axes[2].set_xticklabels(segs, rotation=25, ha='right')
axes[2].set_ylabel('Count'); axes[2].set_title('Customer Count by Segment')

plt.tight_layout()
plt.savefig(PROJECT_ROOT / 'reports/figures/segment_error_analysis.png', dpi=150, bbox_inches='tight')
plt.show(); print('Saved: reports/figures/segment_error_analysis.png')

---
## Section 3 — Worst and Best Predictions

In [ ]:
# Largest underpredictions (actual >> predicted)
top_underpred = test_analysis.nlargest(10, 'residual_gbp')[
    ['actual_gbp', 'predicted_gbp', 'residual_gbp']].reset_index(drop=True)
print('Top 10 Underpredictions (model predicted too LOW):')
print(top_underpred.to_string())
print()
# Largest overpredictions (predicted >> actual)
top_overpred = test_analysis.nsmallest(10, 'residual_gbp')[
    ['actual_gbp', 'predicted_gbp', 'residual_gbp']].reset_index(drop=True)
print('Top 10 Overpredictions (model predicted too HIGH):')
print(top_overpred.to_string())

---
## Section 4 — Heteroscedasticity and Retransformation Bias

In [ ]:
# Test for heteroscedasticity: does error variance grow with predicted value?
pred_quantiles = pd.qcut(test_analysis['predicted_gbp'], q=5, duplicates='drop')
hetero = test_analysis.groupby(pred_quantiles, observed=True)['abs_error_gbp'].mean()
print('Mean absolute error £ by predicted-value quintile:')
for q, val in hetero.items():
    print(f'  {str(q):<30}: £{val:,.2f}')
print()
print('If error grows with predicted value → heteroscedasticity present.')
print()

# Retransformation bias
mean_pred_gbp = test_analysis['predicted_gbp'].mean()
mean_actual_gbp = test_analysis['actual_gbp'].mean()
lognormal_correction = np.exp(residuals_log.mean() + residuals_log.std()**2 / 2)
print(f'Retransformation bias analysis:')
print(f'  Mean actual revenue    : £{mean_actual_gbp:,.2f}')
print(f'  Mean predicted revenue : £{mean_pred_gbp:,.2f}')
print(f'  Ratio (pred/actual)    : {mean_pred_gbp/mean_actual_gbp:.4f}')
print(f'  If ratio < 1: model systematically underpredicts (common with log-target).')
print(f'  Lognormal smearing correction factor: {lognormal_correction:.4f}')

---
## Section 5 — Model Limitations

In [ ]:
print('=== MODEL LIMITATIONS ===')
print()
print('1. LINEAR ASSUMPTION')
print('   The model assumes a linear relationship between standardised features')
print('   and log1p(future_revenue). Non-linear interactions are not captured.')
print()
print('2. HIGH-VALUE CUSTOMER UNDERPREDICTION')
print('   The model systematically underpredicts customers with very high future')
print('   revenue (>£2000). This is a structural limitation of linear models on')
print('   heavily right-skewed targets.')
print()
print('3. LOG1P RETRANSFORMATION BIAS')
print('   E[expm1(y_pred)] != expm1(E[y_pred]) due to Jensen's inequality.')
print('   For revenue-sum forecasting, apply a smearing correction factor.')
print()
print('4. ZERO-REVENUE GROUP')
print('   future_revenue=0 customers (churned) are correctly modelled as a')
print('   group at log1p(0)=0. The model cannot distinguish "will spend £0"')
print('   from "will spend £5" — both map near zero on the log scale.')
print()
print('5. FEATURE SCOPE')
print('   Only RFM-style features are used. Product category, browsing behaviour,')
print('   seasonal patterns, and customer acquisition channel are not available.')
print()
print('6. TEMPORAL EXTRAPOLATION')
print('   The model was trained on the 2010-2011 period. Revenue patterns')
print('   may shift due to external factors not captured in the features.')